<a href="https://colab.research.google.com/github/your-repo/CMSC178IP/blob/main/05%20-%20Noise%20Reduction%20Techniques/notebooks/noise_reduction_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Noise Reduction Techniques Workshop
## CMSC 178IP - Digital Image Processing

**Learning Objectives:**
- Understand different types of image noise and their characteristics
- Implement and compare linear and non-linear noise reduction filters
- Apply advanced techniques like bilateral filtering and morphological operations
- Evaluate denoising performance using quantitative metrics
- Gain hands-on experience with real-world noise reduction scenarios

**Estimated Duration:** 45-60 minutes

## 📦 Setup and Imports

In [ ]:
# Install required packages if running in Colab
import sys
if 'google.colab' in sys.modules:
    !pip install scikit-image opencv-python

import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import ndimage
from skimage import data, filters, restoration
from skimage.util import random_noise
from skimage.filters import threshold_otsu
from skimage.morphology import disk, opening, closing
import warnings
warnings.filterwarnings('ignore')

# Set up matplotlib for better plots
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ Setup complete! Ready to explore noise reduction techniques.")

## 🔍 Part 1: Understanding Image Noise

Let's start by examining different types of noise that commonly affect digital images.

In [ ]:
# Load a test image
original_image = data.camera()
original_image = original_image / 255.0  # Normalize to [0,1]

# Function to add different types of noise
def add_noise_types(image):
    """Add different types of noise to an image"""
    noisy_images = {}
    
    # Gaussian noise
    noisy_images['gaussian'] = random_noise(image, mode='gaussian', var=0.01)
    
    # Salt and pepper noise
    noisy_images['salt_pepper'] = random_noise(image, mode='s&p', amount=0.05)
    
    # Speckle noise (multiplicative)
    noisy_images['speckle'] = random_noise(image, mode='speckle', var=0.01)
    
    # Poisson noise
    noisy_images['poisson'] = random_noise(image, mode='poisson')
    
    return noisy_images

# Generate noisy images
noisy_versions = add_noise_types(original_image)

# Display the results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Types of Image Noise', fontsize=16, fontweight='bold')

# Original
axes[0, 0].imshow(original_image, cmap='gray')
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

# Gaussian noise
axes[0, 1].imshow(noisy_versions['gaussian'], cmap='gray')
axes[0, 1].set_title('Gaussian Noise\n(σ² = 0.01)')
axes[0, 1].axis('off')

# Salt and pepper
axes[0, 2].imshow(noisy_versions['salt_pepper'], cmap='gray')
axes[0, 2].set_title('Salt & Pepper Noise\n(5% pixels)')
axes[0, 2].axis('off')

# Speckle
axes[1, 0].imshow(noisy_versions['speckle'], cmap='gray')
axes[1, 0].set_title('Speckle Noise\n(Multiplicative)')
axes[1, 0].axis('off')

# Poisson
axes[1, 1].imshow(noisy_versions['poisson'], cmap='gray')
axes[1, 1].set_title('Poisson Noise\n(Shot noise)')
axes[1, 1].axis('off')

# Hide last subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("🔍 Observation: Notice how different noise types affect image quality differently:")
print("   • Gaussian noise: Additive, affects all pixels")
print("   • Salt & Pepper: Impulse noise, affects random pixels completely")
print("   • Speckle: Multiplicative, intensity-dependent")
print("   • Poisson: Signal-dependent, common in low-light imaging")

## 🔧 Part 2: Linear Filtering Methods

Linear filters work by replacing each pixel with a weighted combination of its neighbors.

In [ ]:
# Use Gaussian noise for linear filter comparison
noisy_image = noisy_versions['gaussian']

# Apply different linear filters
def apply_linear_filters(image):
    """Apply various linear filters to an image"""
    results = {}
    
    # Mean filter (3x3)
    results['mean_3x3'] = ndimage.uniform_filter(image, size=3)
    
    # Mean filter (5x5)
    results['mean_5x5'] = ndimage.uniform_filter(image, size=5)
    
    # Gaussian filter (σ=1.0)
    results['gaussian_1'] = ndimage.gaussian_filter(image, sigma=1.0)
    
    # Gaussian filter (σ=2.0)
    results['gaussian_2'] = ndimage.gaussian_filter(image, sigma=2.0)
    
    return results

linear_results = apply_linear_filters(noisy_image)

# Display results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Linear Noise Reduction Filters', fontsize=16, fontweight='bold')

# Original noisy
axes[0, 0].imshow(noisy_image, cmap='gray')
axes[0, 0].set_title('Noisy Image\n(Gaussian noise)')
axes[0, 0].axis('off')

# Mean filters
axes[0, 1].imshow(linear_results['mean_3x3'], cmap='gray')
axes[0, 1].set_title('Mean Filter\n(3×3 kernel)')
axes[0, 1].axis('off')

axes[0, 2].imshow(linear_results['mean_5x5'], cmap='gray')
axes[0, 2].set_title('Mean Filter\n(5×5 kernel)')
axes[0, 2].axis('off')

# Gaussian filters
axes[1, 0].imshow(linear_results['gaussian_1'], cmap='gray')
axes[1, 0].set_title('Gaussian Filter\n(σ = 1.0)')
axes[1, 0].axis('off')

axes[1, 1].imshow(linear_results['gaussian_2'], cmap='gray')
axes[1, 1].set_title('Gaussian Filter\n(σ = 2.0)')
axes[1, 1].axis('off')

# Hide last subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("📊 Key Observations:")
print("   • Larger kernels/σ values provide more smoothing but blur edges more")
print("   • Gaussian filters provide smoother results than simple mean filters")
print("   • All linear filters blur edges while reducing noise")

## 🎯 Part 3: Non-Linear Filtering Methods

Non-linear filters can preserve edges better while reducing noise.

In [ ]:
# Use salt & pepper noise for non-linear filter comparison
sp_noisy = noisy_versions['salt_pepper']

# Apply different non-linear filters
def apply_nonlinear_filters(image):
    """Apply various non-linear filters to an image"""
    results = {}
    
    # Median filter (3x3)
    results['median_3x3'] = ndimage.median_filter(image, size=3)
    
    # Median filter (5x5)
    results['median_5x5'] = ndimage.median_filter(image, size=5)
    
    # Maximum filter
    results['maximum'] = ndimage.maximum_filter(image, size=3)
    
    # Minimum filter
    results['minimum'] = ndimage.minimum_filter(image, size=3)
    
    # Compare with Gaussian filter on same noise
    results['gaussian_comparison'] = ndimage.gaussian_filter(image, sigma=1.0)
    
    return results

nonlinear_results = apply_nonlinear_filters(sp_noisy)

# Display results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Non-Linear vs Linear Filters on Salt & Pepper Noise', fontsize=16, fontweight='bold')

# Original noisy
axes[0, 0].imshow(sp_noisy, cmap='gray')
axes[0, 0].set_title('Salt & Pepper\nNoisy Image')
axes[0, 0].axis('off')

# Median filters
axes[0, 1].imshow(nonlinear_results['median_3x3'], cmap='gray')
axes[0, 1].set_title('Median Filter\n(3×3 kernel)')
axes[0, 1].axis('off')

axes[0, 2].imshow(nonlinear_results['median_5x5'], cmap='gray')
axes[0, 2].set_title('Median Filter\n(5×5 kernel)')
axes[0, 2].axis('off')

# Other filters
axes[1, 0].imshow(nonlinear_results['maximum'], cmap='gray')
axes[1, 0].set_title('Maximum Filter\n(3×3 kernel)')
axes[1, 0].axis('off')

axes[1, 1].imshow(nonlinear_results['minimum'], cmap='gray')
axes[1, 1].set_title('Minimum Filter\n(3×3 kernel)')
axes[1, 1].axis('off')

# Gaussian for comparison
axes[1, 2].imshow(nonlinear_results['gaussian_comparison'], cmap='gray')
axes[1, 2].set_title('Gaussian Filter\n(for comparison)')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("🎯 Key Insights:")
print("   • Median filter excels at removing salt & pepper noise")
print("   • Maximum filter removes 'pepper' (dark spots)")
print("   • Minimum filter removes 'salt' (bright spots)")
print("   • Gaussian filter struggles with impulse noise")

## 🚀 Part 4: Advanced Techniques

Modern denoising methods that preserve edges while reducing noise.

In [ ]:
# Bilateral filtering demonstration
def apply_advanced_filters(image):
    """Apply advanced denoising techniques"""
    results = {}
    
    # Convert to uint8 for OpenCV
    image_uint8 = (image * 255).astype(np.uint8)
    
    # Bilateral filter with different parameters
    results['bilateral_1'] = cv2.bilateralFilter(image_uint8, d=9, 
                                               sigmaColor=50, sigmaSpace=50) / 255.0
    
    results['bilateral_2'] = cv2.bilateralFilter(image_uint8, d=9, 
                                               sigmaColor=75, sigmaSpace=75) / 255.0
    
    # Gaussian for comparison
    results['gaussian'] = ndimage.gaussian_filter(image, sigma=1.0)
    
    return results

# Apply to Gaussian noisy image
advanced_results = apply_advanced_filters(noisy_image)

# Display results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Advanced Denoising: Edge-Preserving Filters', fontsize=16, fontweight='bold')

# Original noisy
axes[0, 0].imshow(noisy_image, cmap='gray')
axes[0, 0].set_title('Noisy Image\n(Gaussian noise)')
axes[0, 0].axis('off')

# Bilateral filters
axes[0, 1].imshow(advanced_results['bilateral_1'], cmap='gray')
axes[0, 1].set_title('Bilateral Filter\n(σ_color=50, σ_space=50)')
axes[0, 1].axis('off')

axes[1, 0].imshow(advanced_results['bilateral_2'], cmap='gray')
axes[1, 0].set_title('Bilateral Filter\n(σ_color=75, σ_space=75)')
axes[1, 0].axis('off')

# Gaussian for comparison
axes[1, 1].imshow(advanced_results['gaussian'], cmap='gray')
axes[1, 1].set_title('Gaussian Filter\n(for comparison)')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("🚀 Advanced Filtering Benefits:")
print("   • Bilateral filter preserves edges while smoothing noise")
print("   • σ_color controls how dissimilar colors are averaged")
print("   • σ_space controls the size of the neighborhood")
print("   • Better edge preservation compared to Gaussian filter")

## 📊 Part 5: Performance Evaluation

Quantitative metrics to assess denoising performance.

In [ ]:
# Performance evaluation functions
def calculate_psnr(original, denoised):
    """Calculate Peak Signal-to-Noise Ratio"""
    mse = np.mean((original - denoised) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0  # Assuming normalized images
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

def calculate_mse(original, denoised):
    """Calculate Mean Squared Error"""
    return np.mean((original - denoised) ** 2)

def calculate_ssim_simple(original, denoised):
    """Simplified SSIM calculation"""
    mu1 = np.mean(original)
    mu2 = np.mean(denoised)
    sigma1 = np.var(original)
    sigma2 = np.var(denoised)
    sigma12 = np.mean((original - mu1) * (denoised - mu2))
    
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    
    ssim = ((2 * mu1 * mu2 + c1) * (2 * sigma12 + c2)) / \
           ((mu1**2 + mu2**2 + c1) * (sigma1 + sigma2 + c2))
    return ssim

# Evaluate different methods on Gaussian noise
methods = {
    'Noisy': noisy_image,
    'Mean (3×3)': linear_results['mean_3x3'],
    'Gaussian (σ=1)': linear_results['gaussian_1'],
    'Median (3×3)': apply_nonlinear_filters(noisy_image)['median_3x3'],
    'Bilateral': advanced_results['bilateral_1']
}

# Calculate metrics
metrics_data = []
for name, result in methods.items():
    psnr = calculate_psnr(original_image, result)
    mse = calculate_mse(original_image, result)
    ssim = calculate_ssim_simple(original_image, result)
    metrics_data.append([name, psnr, mse, ssim])

# Display metrics
print("📊 PERFORMANCE METRICS COMPARISON")
print("=" * 60)
print(f"{'Method':<15} {'PSNR (dB)':<12} {'MSE':<12} {'SSIM':<8}")
print("-" * 60)
for name, psnr, mse, ssim in metrics_data:
    print(f"{name:<15} {psnr:<12.2f} {mse:<12.6f} {ssim:<8.3f}")

# Visualize metrics
methods_names = [data[0] for data in metrics_data]
psnr_values = [data[1] for data in metrics_data]
ssim_values = [data[3] for data in metrics_data]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PSNR comparison
bars1 = ax1.bar(methods_names, psnr_values, color=['red', 'blue', 'green', 'orange', 'purple'])
ax1.set_title('Peak Signal-to-Noise Ratio (PSNR)', fontweight='bold')
ax1.set_ylabel('PSNR (dB)')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# Add value labels
for bar, value in zip(bars1, psnr_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{value:.1f}', ha='center', va='bottom')

# SSIM comparison
bars2 = ax2.bar(methods_names, ssim_values, color=['red', 'blue', 'green', 'orange', 'purple'])
ax2.set_title('Structural Similarity Index (SSIM)', fontweight='bold')
ax2.set_ylabel('SSIM')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

# Add value labels
for bar, value in zip(bars2, ssim_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{value:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n💡 Metric Interpretation:")
print("   • Higher PSNR = better quality (less distortion)")
print("   • Higher SSIM = better structural similarity")
print("   • Lower MSE = less error")
print("   • Best method depends on the application and noise type")

## 🧪 Part 6: Student Activity (15 minutes)

**Your Task:** Implement and compare noise reduction techniques for medical imaging!

**Scenario:** You're working with a medical imaging system that produces noisy X-ray images. Your goal is to find the best denoising approach that preserves important diagnostic details while reducing noise.

In [ ]:
# Create a synthetic medical image
def create_medical_phantom():
    """Create a simplified medical phantom image"""
    phantom = np.zeros((256, 256))
    
    # Create circular regions (organs)
    center = (128, 128)
    y, x = np.ogrid[:256, :256]
    
    # Main organ (lung-like)
    main_organ = (x - center[0])**2 + (y - center[1])**2 < 80**2
    phantom[main_organ] = 0.6
    
    # Dense region (bone-like)
    dense_region = (x - 150)**2 + (y - 100)**2 < 25**2
    phantom[dense_region] = 0.9
    
    # Small details (vessels)
    for i in range(50, 200, 20):
        vessel = (x - i)**2 + (y - 120)**2 < 3**2
        phantom[vessel] = 0.3
    
    return phantom

# Create medical phantom and add noise
medical_phantom = create_medical_phantom()
medical_noisy = random_noise(medical_phantom, mode='gaussian', var=0.02)

# Display the problem
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.imshow(medical_phantom, cmap='gray')
ax1.set_title('Clean Medical Image\n(Ground Truth)')
ax1.axis('off')

ax2.imshow(medical_noisy, cmap='gray')
ax2.set_title('Noisy Medical Image\n(Your task: denoise this!)')
ax2.axis('off')

plt.tight_layout()
plt.show()

print("🧪 YOUR CHALLENGE:")
print("1. Apply at least 3 different denoising methods to the noisy medical image")
print("2. Calculate PSNR and SSIM for each method")
print("3. Determine which method best preserves important medical details")
print("4. Consider: What makes a good denoising method for medical imaging?")
print("\n💡 Hints:")
print("   • Medical images need edge preservation for accurate diagnosis")
print("   • Small details (like vessels) are crucial")
print("   • Consider the trade-off between noise reduction and detail preservation")

In [ ]:
# YOUR CODE HERE - Implement your denoising solutions

# Method 1: [Add your method name]
# method1_result = 

# Method 2: [Add your method name]
# method2_result = 

# Method 3: [Add your method name]
# method3_result = 

# Calculate metrics for each method
# method1_psnr = calculate_psnr(medical_phantom, method1_result)
# method1_ssim = calculate_ssim_simple(medical_phantom, method1_result)

# Display your results
# fig, axes = plt.subplots(2, 2, figsize=(12, 10))
# ...

print("✨ Implement your solution above!")
print("   Try different combinations of filters and parameters")
print("   Compare both visual quality and quantitative metrics")

<details>
<summary><b>🔍 Click here to reveal the solution</b></summary>

```python
# SOLUTION - Medical Image Denoising

# Method 1: Gaussian Filter
method1_result = ndimage.gaussian_filter(medical_noisy, sigma=1.0)
method1_name = "Gaussian Filter (σ=1.0)"

# Method 2: Bilateral Filter
medical_uint8 = (medical_noisy * 255).astype(np.uint8)
method2_result = cv2.bilateralFilter(medical_uint8, d=9, sigmaColor=50, sigmaSpace=50) / 255.0
method2_name = "Bilateral Filter"

# Method 3: Median Filter
method3_result = ndimage.median_filter(medical_noisy, size=3)
method3_name = "Median Filter (3×3)"

# Method 4: Anisotropic Diffusion (simplified)
method4_result = medical_noisy.copy()
for _ in range(5):
    method4_result = ndimage.gaussian_filter(method4_result, sigma=0.5)
method4_name = "Anisotropic Diffusion (simplified)"

# Calculate metrics
methods_medical = {
    'Noisy': medical_noisy,
    method1_name: method1_result,
    method2_name: method2_result,
    method3_name: method3_result,
    method4_name: method4_result
}

print("🏥 MEDICAL IMAGE DENOISING RESULTS")
print("=" * 50)
print(f"{'Method':<25} {'PSNR':<8} {'SSIM':<8}")
print("-" * 50)

for name, result in methods_medical.items():
    psnr = calculate_psnr(medical_phantom, result)
    ssim = calculate_ssim_simple(medical_phantom, result)
    print(f"{name:<25} {psnr:<8.2f} {ssim:<8.3f}")

# Display results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Medical Image Denoising Comparison', fontsize=16, fontweight='bold')

results_list = list(methods_medical.items())
titles = [name for name, _ in results_list]
images = [img for _, img in results_list]

for i, (title, img) in enumerate(zip(titles, images)):
    ax = axes[i//3, i%3]
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

# Hide last subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("\n🎯 Key Findings:")
print("   • Bilateral filter typically provides best balance for medical images")
print("   • Preserves edges (important for diagnosis) while reducing noise")
print("   • Gaussian filter smooths too much, losing critical details")
print("   • Median filter good for impulse noise but may blur fine structures")
```
</details>

## 📚 Part 7: Real-World Applications

Understanding how noise reduction is applied in different domains.

In [ ]:
# Demonstrate domain-specific considerations
print("🌍 REAL-WORLD NOISE REDUCTION APPLICATIONS")
print("=" * 60)

applications = {
    "📱 Smartphone Photography": {
        "challenges": ["Low light conditions", "Small sensor size", "Real-time processing"],
        "solutions": ["Multi-frame averaging", "AI-based denoising", "Bilateral filtering"],
        "metrics": "Visual quality, processing speed"
    },
    "🏥 Medical Imaging": {
        "challenges": ["Low radiation dose", "High accuracy requirements", "Edge preservation"],
        "solutions": ["Anisotropic diffusion", "Non-local means", "Total variation"],
        "metrics": "SNR, edge preservation, diagnostic accuracy"
    },
    "🛰️ Satellite Imaging": {
        "challenges": ["Atmospheric interference", "Long transmission", "Large data volumes"],
        "solutions": ["Wiener filtering", "Kalman filtering", "Wavelet denoising"],
        "metrics": "PSNR, feature preservation, compression ratio"
    },
    "🏭 Industrial Inspection": {
        "challenges": ["Variable lighting", "Real-time requirements", "Defect detection"],
        "solutions": ["Morphological filtering", "Adaptive thresholding", "Median filtering"],
        "metrics": "Detection accuracy, false positive rate, speed"
    }
}

for app_name, details in applications.items():
    print(f"\n{app_name}")
    print(f"  Challenges: {', '.join(details['challenges'])}")
    print(f"  Solutions: {', '.join(details['solutions'])}")
    print(f"  Key Metrics: {details['metrics']}")

print("\n💡 Key Takeaways:")
print("   • No single method works best for all applications")
print("   • Consider domain-specific constraints (speed, accuracy, etc.)")
print("   • Combine multiple techniques for optimal results")
print("   • Validate with appropriate metrics for your application")

## 🎯 Summary and Key Takeaways

Congratulations! You've completed the noise reduction techniques workshop. Here's what you've learned:

In [ ]:
print("🎓 WORKSHOP SUMMARY")
print("=" * 50)

summary_points = [
    "🔍 Noise Types: Gaussian, salt & pepper, speckle, and Poisson noise have different characteristics",
    "⚖️ Linear vs Non-linear: Linear filters blur edges; non-linear filters can preserve them",
    "🎯 Method Selection: Choose filters based on noise type and application requirements",
    "📊 Performance Metrics: PSNR, SSIM, and MSE provide quantitative evaluation",
    "🚀 Advanced Techniques: Bilateral filtering provides excellent edge preservation",
    "🏥 Medical Applications: Edge preservation is crucial for diagnostic accuracy",
    "⚡ Trade-offs: Balance between noise reduction and detail preservation",
    "🌍 Real-world Considerations: Different domains have different constraints and metrics"
]

for i, point in enumerate(summary_points, 1):
    print(f"{i}. {point}")

print("\n🔬 NEXT STEPS:")
print("   • Experiment with different noise types and filter combinations")
print("   • Try implementing your own adaptive filtering algorithms")
print("   • Explore deep learning approaches to image denoising")
print("   • Apply these techniques to your own image processing projects")

print("\n✨ Congratulations on completing the Noise Reduction Techniques workshop!")
print("   You now have the skills to tackle real-world image denoising challenges.")

## 📖 Additional Resources

- **OpenCV Documentation**: [Image Filtering](https://docs.opencv.org/master/d4/d86/group__imgproc__filter.html)
- **Scikit-image**: [Restoration module](https://scikit-image.org/docs/stable/api/skimage.restoration.html)
- **Research Papers**: 
  - Tomasi & Manduchi (1998): "Bilateral filtering for gray and color images"
  - Buades et al. (2005): "A non-local algorithm for image denoising"
- **Books**: 
  - Gonzalez & Woods: "Digital Image Processing" (Chapter 5)
  - Russ: "The Image Processing Handbook" (Chapter 4)